# МТС · Qwen3-0.6B · LoRA на 512 звонках

Это отдельный эксперимент участника команды. Нужен файл **mts_friend_qwen06_payload.zip**, лежащий рядом с блокнотом в переданном архиве. Загрузка весов происходит с официального Hugging Face в Colab, не на ваш компьютер.

Данные: очищенная официальная v2; 512 train / 20 inner-dev. Старый синтетический датасет и официальный val383 здесь не используются. Результатов обучения в этом блокноте ещё нет.

Выполняйте ячейки сверху вниз. Установите «Среда выполнения → Сменить среду выполнения → GPU». **RUN_TRAINING по умолчанию False** — его нужно включить перед единственным обучающим запуском.


## 1. GPU, загрузка одного payload и проверка файлов

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, subprocess, sys, zipfile
import torch
from google.colab import files

assert torch.cuda.is_available(), 'Выберите GPU в настройках среды Colab; обучение на CPU не запускается'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('Torch:', torch.__version__)
print('CUDA:', torch.version.cuda)
uploaded = files.upload()
PAYLOAD_NAME = 'mts_friend_qwen06_payload.zip'
assert PAYLOAD_NAME in uploaded, 'Загрузите именно mts_friend_qwen06_payload.zip'
EXPECTED_PAYLOAD_SHA = '39cd9de9ba4ec62a40fe75010375014ed8aff7ac5d03a32debd55bc986f87733'
assert hashlib.sha256(uploaded[PAYLOAD_NAME]).hexdigest() == EXPECTED_PAYLOAD_SHA, 'ZIP отличается от выданного пакета'
ROOT = Path('/content/mts_friend_qwen06')
if ROOT.exists() and any(ROOT.iterdir()):
    raise RuntimeError('Payload уже распакован. Продолжайте со следующей ячейки; не перезаписывайте его')
with zipfile.ZipFile(PAYLOAD_NAME) as z:
    assert z.testzip() is None
    assert sum(item.file_size for item in z.infolist()) < 100 * 1024**2
    for item in z.infolist():
        path = (Path('/content') / item.filename).resolve()
        assert path.is_relative_to(ROOT.resolve())
        assert (item.external_attr >> 16) & 0o170000 != 0o120000
    z.extractall('/content')
del uploaded
manifest = json.loads((ROOT/'manifest.json').read_text(encoding='utf-8'))
for rel, expected in manifest['sha256'].items():
    path = (ROOT/rel).resolve()
    assert path.is_relative_to(ROOT.resolve()) and hashlib.sha256(path.read_bytes()).hexdigest() == expected, rel
print('SHA256 файлов проверены. Выборки:', manifest['counts'])


## 2. Зависимости и локальные проверки

Сохраняем CUDA PyTorch среды. TRL/datasets/pyarrow/bitsandbytes для этого опыта не нужны. Скрипт проверки не загружает веса: он сверяет разметку, маски, EOS, длины, изоляцию данных и одинаковые сообщения при обучении и генерации.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT/'requirements.txt')], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; print("Torch",torch.__version__,"Transformers",transformers.__version__,"PEFT",peft.__version__); from transformers import AutoModelForCausalLM'], check=True)
subprocess.run([sys.executable, str(ROOT/'verify_friend.py'), '--tokenizer'], cwd=ROOT, check=True)


## 3. Папка результатов

По умолчанию файлы сохраняются в сессионном хранилище Colab. Его можно потерять при отключении: скачайте ZIP результата. Для сохранения checkpoint в своём Google Drive установите SAVE_TO_DRIVE=True и сами подтвердите подключение. Веса и данные в публичный GitHub не загружаются.


In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS = Path('/content/drive/MyDrive/mts_qwen06_runs')
else:
    RUNS = Path('/content/mts_friend_runs')
RUN = RUNS / datetime.now(timezone.utc).strftime('qwen06_512_%Y%m%dT%H%M%SZ')
PRECISION = 'float16'
LEARNING_RATE = 0.0001
BUDGET_HOURS = 4
print('Новая папка эксперимента:', RUN)
print('FP16 base, новая LoRA r16/alpha32, 512 звонков, один проход. Финальная оценка занимает дополнительное время.')


## 4. Один запуск обучения

Установите RUN_TRAINING=True. Первые три обновления должны быть конечными и менять LoRA; ошибка останавливает процесс. Last сохраняется каждые 32 шага; best выбирается по извлечению на фиксированных 4dev. После обучения — свежая загрузка base+adapter и проверка всех 20dev.

Не запускайте эту ячейку второй раз в ту же папку. При ошибке архив с доступными артефактами скачивается в finally; инструкция продолжения находится в CODEX_TASK.md. Полное обучение 3375 автоматически не запускается.


In [ ]:
RUN_TRAINING = False

def download_current_result():
    if not RUN.exists():
        print('Папки результата пока нет')
        return
    code_zip = RUN/'executed_source.zip'
    with zipfile.ZipFile(code_zip, 'w', zipfile.ZIP_DEFLATED) as z:
        for rel in manifest['sha256']:
            if rel.endswith('.py') or rel in ('requirements.txt', 'experiment_defaults.json', 'selection.json'):
                z.write(ROOT/rel, rel)
        z.write(ROOT/'manifest.json', 'payload_manifest.json')
    result_zip = shutil.make_archive('/content/' + RUN.name + '_results', 'zip', RUN)
    print('Скачайте результат до отключения среды:', result_zip)
    files.download(result_zip)

if RUN_TRAINING:
    assert not RUN.exists(), 'Выберите новую папку для нового опыта; существующий запуск не перезаписывается'
    command = [sys.executable, '-u', '-m', 'ml.qwen.friend_qwen06', '--root', str(ROOT), '--out', str(RUN),
               '--precision', PRECISION, '--learning-rate', str(LEARNING_RATE), '--budget-hours', str(BUDGET_HOURS)]
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding='utf-8', errors='replace', bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        download_current_result()
else:
    print('Обучение не запускалось. Для единственного запуска установите RUN_TRAINING=True.')


## 5. Отчёт и разбор ошибок

Передайте команде весь скачанный архив. before_dev4 и after_dev4_same_ids — парное сравнение на одинаковых ID. after_dev20 — отдельная, более широкая оценка. Эти 20 звонков не заменяют официальную проверку 383.


In [ ]:
report_path = RUN/'experiment_report.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps(report, ensure_ascii=False, indent=2))
    if (RUN/'after_dev20/predictions.jsonl').exists():
        subprocess.run([sys.executable, '-m', 'ml.qwen.analyze_results', '--gold', str(ROOT/'data/dev_20.jsonl'),
            '--prediction-dir', str(RUN/'after_dev20'), '--evaluator', str(ROOT/'evaluation/case_metrics.py'),
            '--output', str(RUN/'dev_error_analysis.json')], cwd=ROOT, check=True)
    else:
        print('Финальная оценка не выполнена; статус остановки указан в отчёте')
elif (RUN/'failure.json').exists():
    print((RUN/'failure.json').read_text(encoding='utf-8'))
else:
    print('Отчёта ещё нет: обучение не запускалось или не завершено')


## 6. Повторно скачать архив с разбором ошибок
Выполните после предыдущей ячейки. Также подходит для сохранения доступных результатов после ошибки.

In [ ]:
if RUN.exists():
    download_current_result()
else:
    print('Нечего скачивать: запуск ещё не создан')


## 7. Необязательная проверка сохранённого адаптера

Флаг выключен: «Выполнить всё» не начинает дополнительную генерацию. Recheck только оценивает сохранённые веса; он не продолжает обучение и не возобновляет optimizer. При новой сессии сначала восстановите ROOT и RUN из скачанных архивов. Контрольные веса 0.8B сюда не подходят.


In [ ]:
RUN_RECHECK = False
ADAPTER_TO_CHECK = 'last'
if RUN_RECHECK:
    subprocess.run([sys.executable, '-u', '-m', 'ml.qwen.friend_qwen06', '--root', str(ROOT), '--out', str(RUN),
                    '--precision', PRECISION, '--recheck', ADAPTER_TO_CHECK], cwd=ROOT, check=True)
else:
    print('Повторная оценка не запускалась')
